In [1]:
# -*- coding: utf-8 -*-
"""
reporting.py

Este script gera os componentes visuais e tabulares para o relatório final de
análise de risco climático, focando no estudo de caso do Rio Grande do Sul.
Ele carrega o GeoDataFrame final do pipeline e produz uma série de mapas,
gráficos e tabelas como saída.
Versão corrigida para resolver o TypeError na criação de legendas.
"""

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import os
import logging

from google.colab import drive

drive.mount('/content/drive/',force_remount=True)

# --- Configurações ---
# INPUT_GEOPARQUET_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.geoparquet'
# OUTPUT_REPORT_DIR = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/04_report_assets/'
# STATE_TO_ANALYZE = 'RS'
# TOP_N_MUNICIPALITIES = 10 # Número de municípios a serem listados na tabela de maior risco
# --------------------------------------------------------------------------

logging.basicConfig(filename='reporting_log.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Configurações de estilo para os gráficos
sns.set_theme(style="whitegrid", rc={
    "axes.facecolor": "#f0f2f5",
    "grid.color": "#ffffff",
    "figure.facecolor": "#ffffff",
    "axes.edgecolor": "#cccccc",
    "axes.labelcolor": "#333333",
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "text.color": "#333333",
})

def plot_geospatial_map(gdf, column, title, filename, colormap='viridis', categorical=False, legend_kwds=None):
    """
    Gera e salva um mapa coroplético a partir de um GeoDataFrame.

    Args:
        gdf (gpd.GeoDataFrame): Dados a serem plotados.
        column (str): Nome da coluna para usar na simbologia.
        title (str): Título do mapa.
        filename (str): Nome do arquivo para salvar a imagem.
        colormap (str or dict): Colormap para usar.
        categorical (bool): Se a variável é categórica.
        legend_kwds (dict, optional): Argumentos para a legenda.
    """
    logging.info(f"Gerando mapa para: {column}")
    print(f"Gerando mapa para: {column}...")

    fig, ax = plt.subplots(1, 1, figsize=(12, 12))

    # *** CORREÇÃO DO ERRO ***
    # Define as palavras-chave da legenda padrão se não forem fornecidas.
    # O argumento 'orientation' foi removido pois não é válido para legendas categóricas.
    # Usamos 'loc' e 'bbox_to_anchor' para posicionar a legenda fora do mapa.
    if legend_kwds is None:
        legend_kwds = {'loc': 'upper left', 'bbox_to_anchor': (1, 1)}

    gdf.plot(column=column,
             ax=ax,
             legend=True,
             legend_kwds=legend_kwds,
             cmap=colormap,
             categorical=categorical,
             edgecolor='0.8',
             linewidth=0.5)

    ax.set_title(title, fontdict={'fontsize': '16', 'fontweight': '500'})
    ax.set_axis_off()

    output_dir = os.path.dirname(filename)
    os.makedirs(output_dir, exist_ok=True)

    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    logging.info(f"Mapa salvo em: {filename}")
    print(f"--> Mapa salvo em: {filename}")


def plot_bar_chart(data, x_col, y_col, title, filename, xlabel, ylabel, palette='viridis'):
    """Gera e salva um gráfico de barras."""
    logging.info(f"Gerando gráfico de barras para: {title}")
    print(f"Gerando gráfico de barras para: {title}...")

    plt.figure(figsize=(10, 6))
    sns.barplot(x=x_col, y=y_col, data=data, palette=palette)
    plt.title(title, fontsize=16)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.tight_layout()

    output_dir = os.path.dirname(filename)
    os.makedirs(output_dir, exist_ok=True)

    plt.savefig(filename, dpi=300)
    plt.close()
    logging.info(f"Gráfico salvo em: {filename}")
    print(f"--> Gráfico salvo em: {filename}")


def main_reporting(config):
    """
    Função principal para executar o pipeline de geração de relatórios.
    """
    logging.info("--- Iniciando Pipeline de Geração de Relatórios ---")
    print("--- Iniciando Pipeline de Geração de Relatórios ---")

    input_path = config['INPUT_GEOPARQUET_PATH']
    output_dir = config['OUTPUT_REPORT_DIR']
    state = config['STATE_TO_ANALYZE']

    os.makedirs(output_dir, exist_ok=True)

    # 1. Carregar e Filtrar Dados
    logging.info(f"Carregando dados de: {input_path}")
    print(f"Carregando dados de: {input_path}...")
    try:
        gdf_full = gpd.read_parquet(input_path)
    except Exception as e:
        print(f"ERRO: Não foi possível carregar o arquivo {input_path}. Erro: {e}")
        logging.error(f"Não foi possível carregar o arquivo {input_path}: {e}")
        return

    gdf = gdf_full[gdf_full['SIGLA_UF'] == state].copy()
    if gdf.empty:
        print(f"ERRO: Nenhum dado encontrado para o estado '{state}'. Verifique o arquivo de entrada.")
        logging.error(f"Nenhum dado encontrado para o estado '{state}'.")
        return
    print(f"Dados filtrados para o estado: {state}. Total de {len(gdf)} municípios.")

    # 2. Gerar Mapas
    # Mapa de Risco MCDA
    risk_cat_order = ['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto']
    risk_colors = ['#2c7bb6', '#abd9e9', '#ffffbf', '#fdae61', '#d7191c']
    risk_cmap = mcolors.ListedColormap(risk_colors)
    gdf['Risco_Ampliado_MCDA_Cat'] = pd.Categorical(gdf['Risco_Ampliado_MCDA_Cat'], categories=risk_cat_order, ordered=True)
    plot_geospatial_map(gdf, 'Risco_Ampliado_MCDA_Cat', f'Score de Risco MCDA - {state}', os.path.join(output_dir, 'map_risco_mcda.png'), colormap=risk_cmap, categorical=True)

    # Mapa de Tendência
    trend_cat_order = ['Decrescente', 'Estável', 'Crescente']
    trend_colors = ['#1a9641', '#ffffbf', '#d7191c']
    trend_cmap = mcolors.ListedColormap(trend_colors)
    gdf['Tendencia_Eventos_Climaticos_Extremos'] = pd.Categorical(gdf['Tendencia_Eventos_Climaticos_Extremos'], categories=trend_cat_order, ordered=True)
    plot_geospatial_map(gdf, 'Tendencia_Eventos_Climaticos_Extremos', f'Tendência de Eventos Climáticos - {state}', os.path.join(output_dir, 'map_tendencia.png'), colormap=trend_cmap, categorical=True)

    # Mapa de Principal Ameaça
    plot_geospatial_map(gdf, 'principal_ameaca', f'Principal Ameaça Geohidrológica - {state}', os.path.join(output_dir, 'map_principal_ameaca.png'), colormap='tab20', categorical=True)

    # Mapa de Clusters LISA
    lisa_labels_col = 'LAST10_YEARS_COUNT_lisa_labels'
    if lisa_labels_col in gdf.columns:
        lisa_cat_order = ['0 ns (Não Significativo)', '1 HH (Hot Spot)', '2 LH (Anel Baixo-Alto)', '3 LL (Cold Spot)', '4 HL (Anel Alto-Baixo)']
        lisa_colors = ['#eeeeee', '#d7191c', '#fdae61', '#2c7bb6', '#abd9e9']
        lisa_cmap = mcolors.ListedColormap(lisa_colors)
        gdf[lisa_labels_col] = pd.Categorical(gdf[lisa_labels_col], categories=lisa_cat_order, ordered=True)
        plot_geospatial_map(gdf, lisa_labels_col, f'Clusters Espaciais (LISA) - {state}', os.path.join(output_dir, 'map_lisa_clusters.png'), colormap=lisa_cmap, categorical=True)
    else:
        print(f"AVISO: Coluna de clusters LISA '{lisa_labels_col}' não encontrada. Mapa não gerado.")

    # 3. Gerar Gráficos
    # Gráfico de Distribuição de Risco
    risk_dist_data = gdf['Risco_Ampliado_MCDA_Cat'].value_counts().reindex(risk_cat_order).reset_index()
    risk_dist_data.columns = ['Categoria de Risco', 'Número de Municípios']
    plot_bar_chart(risk_dist_data, 'Número de Municípios', 'Categoria de Risco', f'Distribuição por Categoria de Risco - {state}', os.path.join(output_dir, 'chart_distribuicao_risco.png'), 'Número de Municípios', 'Categoria de Risco')

    # Gráfico de Tendência
    trend_dist_data = gdf['Tendencia_Eventos_Climaticos_Extremos'].value_counts().reindex(trend_cat_order).reset_index()
    trend_dist_data.columns = ['Tendência', 'Número de Municípios']
    plot_bar_chart(trend_dist_data, 'Número de Municípios', 'Tendência', f'Distribuição por Tendência - {state}', os.path.join(output_dir, 'chart_distribuicao_tendencia.png'), 'Número de Municípios', 'Tendência')

    # Gráfico de Principais Ameaças
    threat_dist_data = gdf['principal_ameaca'].value_counts().nlargest(5).reset_index()
    threat_dist_data.columns = ['Ameaça Principal', 'Número de Municípios']
    plot_bar_chart(threat_dist_data, 'Número de Municípios', 'Ameaça Principal', f'Top 5 Ameaças Dominantes - {state}', os.path.join(output_dir, 'chart_top_ameacas.png'), 'Número de Municípios', 'Ameaça Principal')

    # 4. Gerar Tabela de Destaques
    top_n = config.get('TOP_N_MUNICIPALITIES', 10)
    logging.info(f"Gerando tabela com Top {top_n} municípios de maior risco.")
    print(f"Gerando tabela com Top {top_n} municípios de maior risco...")

    cols_to_show = [
        'NM_MUN',
        'Risco_Ampliado_MCDA_Score',
        'Risco_Ampliado_MCDA_Cat',
        'Tendencia_Eventos_Climaticos_Extremos',
        'principal_ameaca',
        'HISTORIC_COUNT',
        'CENSO_2020_POP'
    ]
    # Filtrar para colunas que realmente existem
    cols_to_show = [c for c in cols_to_show if c in gdf.columns]

    summary_table = gdf.sort_values(by='Risco_Ampliado_MCDA_Score', ascending=False).head(top_n)
    summary_table = summary_table[cols_to_show]
    table_path = os.path.join(output_dir, f'tabela_top_{top_n}_risco.csv')
    summary_table.to_csv(table_path, index=False, sep=';')
    logging.info(f"Tabela de resumo salva em: {table_path}")
    print(f"--> Tabela de resumo salva em: {table_path}")

    print("--- Pipeline de Geração de Relatórios Concluído ---")


if __name__ == '__main__':
    BASE_PROJECT_PATH = '/content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr'

    config_reporting = {
        'INPUT_GEOPARQUET_PATH': os.path.join(BASE_PROJECT_PATH, '03_gold/climate_risk_final_analysis.geoparquet'),
        'OUTPUT_REPORT_DIR': os.path.join(BASE_PROJECT_PATH, '04_report_assets/'),
        'STATE_TO_ANALYZE': 'RS',
        'TOP_N_MUNICIPALITIES': 10
    }

    # Descomente a linha abaixo para executar o script
    main_reporting(config_reporting)
    print("Exemplo de execução do main_reporting (comentado).")
    print("Certifique-se que o arquivo de entrada final existe.")
    print("Descomente a linha `main_reporting(config_reporting)` para executar.")

Mounted at /content/drive/
--- Iniciando Pipeline de Geração de Relatórios ---
Carregando dados de: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/03_gold/climate_risk_final_analysis.geoparquet...
Dados filtrados para o estado: RS. Total de 499 municípios.
Gerando mapa para: Risco_Ampliado_MCDA_Cat...
--> Mapa salvo em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/04_report_assets/map_risco_mcda.png
Gerando mapa para: Tendencia_Eventos_Climaticos_Extremos...
--> Mapa salvo em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/04_report_assets/map_tendencia.png
Gerando mapa para: principal_ameaca...
--> Mapa salvo em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/04_report_assets/map_principal_ameaca.png
Gerando mapa para: LAST10_YEARS_COUNT_lisa_labels...
--> Mapa salvo em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/04_report_assets/map_lisa_clusters.png
Gerando g

/tmp/ipykernel_2138/1462376513.py:98: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=x_col, y=y_col, data=data, palette=palette)


--> Gráfico salvo em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/04_report_assets/chart_distribuicao_risco.png
Gerando gráfico de barras para: Distribuição por Tendência - RS...


/tmp/ipykernel_2138/1462376513.py:98: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=x_col, y=y_col, data=data, palette=palette)


--> Gráfico salvo em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/04_report_assets/chart_distribuicao_tendencia.png
Gerando gráfico de barras para: Top 5 Ameaças Dominantes - RS...


/tmp/ipykernel_2138/1462376513.py:98: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=x_col, y=y_col, data=data, palette=palette)


--> Gráfico salvo em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/04_report_assets/chart_top_ameacas.png
Gerando tabela com Top 10 municípios de maior risco...
--> Tabela de resumo salva em: /content/drive/MyDrive/workspace/skyvidya/00_pocs_ingestao_ciencia/mdr/04_report_assets/tabela_top_10_risco.csv
--- Pipeline de Geração de Relatórios Concluído ---
Exemplo de execução do main_reporting (comentado).
Certifique-se que o arquivo de entrada final existe.
Descomente a linha `main_reporting(config_reporting)` para executar.
